A `GroupingStrategy` (`qarp.operators`) partitions the Pauli terms of an operator into groups that can be measured — or exponentiated — together. Strategies are *pure index partitioners*: they return groups of indices, never touch coefficients, and leave the per-group diagonalisation to the consumer. Three ship with qarp:

- `NoGrouping` — one term per group (termwise behaviour)
- `QubitWiseCommuting` — greedy grouping under qubit-wise commutation (diagonalisable with single-qubit rotations only)
- `FullyCommuting` — greedy grouping under general commutation (fewest groups; needs an entangling Clifford, derived in C++)

This notebook shows:
1. Partitioning an operator's terms directly
2. Grouping in measurement — `PauliAveraging` circuit counts
3. Grouping in time evolution — `TrotterBlock` and symmetry conservation
4. Writing a custom strategy

In [ ]:
import numpy as np
import qarp

from qarp.operators import QubitOperator
from qarp.operators import (
    FullyCommuting,
    GroupingStrategy,
    NoGrouping,
    QubitWiseCommuting,
    group_basis,
)
from qarp.blocks import CompositeBlock, ComputationalBasisStateBlock
from qarp.blocks import HEABlock, TrotterBlock
from qarp.algorithms import PauliAveraging, Sampler, StateVector
from qarp.engines import QarpEngine

## 1. Partitioning an operator directly

`strategy.group(terms, n_qubits)` takes sparse Pauli terms (`{qubit: 'X'|'Y'|'Z'}`) and returns an index partition. The classic example: `X0 X1`, `Y0 Y1` and `Z0 Z1` all *commute* with each other, but no two of them are *qubit-wise* commuting — so `FullyCommuting` merges them where `QubitWiseCommuting` cannot.

In [ ]:
H2q = (QubitOperator("Z0", 0.4) + QubitOperator("Z1", -0.3) + QubitOperator("X0 X1", 0.25)
       + QubitOperator("Y0 Y1", 0.25) + QubitOperator("Z0 Z1", 0.7))

terms = [dict(t) for t in H2q.terms if t]  # sparse {qubit: letter} dicts, identity dropped
labels = [" ".join(f"{p}{q}" for q, p in sorted(t.items())) for t in terms]

for strategy in (NoGrouping(), QubitWiseCommuting(), FullyCommuting()):
    groups = strategy.group(terms, 2)
    pretty = [[labels[i] for i in g] for g in groups]
    print(f"{strategy!r:24} {len(groups)} groups: {pretty}")

For a qubit-wise commuting group, every term agrees on each shared qubit, so a per-qubit basis recipe diagonalises the whole group (`H` for X, `Sdg·H` for Y — no entangling gates). `group_basis` returns that recipe; strategies advertise this guarantee via the `qubit_wise` class flag, which consumers that cannot insert an entangling Clifford (circuit cutting) check.

In [ ]:
qwc_groups = QubitWiseCommuting().group(terms, 2)
print("measurement basis per QWC group:")
for g in qwc_groups:
    print(f"  {[labels[i] for i in g]!s:30} -> {group_basis(g, terms)}")
print("qubit_wise flags:", {type(s).__name__: s.qubit_wise
                            for s in (NoGrouping(), QubitWiseCommuting(), FullyCommuting())})

## 2. Grouping in measurement — `PauliAveraging`

`PauliAveraging(grouping=...)` builds one measurement circuit per group (default: `FullyCommuting`). Fewer groups means fewer circuits and fewer shot budgets — the estimate itself is unbiased for any strategy. Here we use `n_shots=qarp.EXACT` (exact probabilities) so all three strategies must agree to machine precision with the `StateVector` reference.

In [ ]:
Hfull = QubitOperator()
for i in range(3):
    Hfull += QubitOperator(f"X{i} X{i+1}", 0.5)
    Hfull += QubitOperator(f"Y{i} Y{i+1}", 0.5)
Hfull += QubitOperator("Z0 Z1", 0.3) + QubitOperator("Z1 Z2", 0.3) + QubitOperator("Z0", 0.1)

def hea():
    return HEABlock(4, n_layers=1, real=True, linear=True, circular=False, use_cz=False).build()

def params_for(block):
    return dict(zip(block.symbols, np.linspace(0.1, 0.9, len(block.symbols))))

ref = hea()
sv = StateVector(operator=Hfull, ket=ref)
engine = QarpEngine()
engine.build([sv])
print(f"StateVector reference   <H> = {engine.run(params_for(ref))[0].real:+.12f}")

for strategy in (NoGrouping(), QubitWiseCommuting(), FullyCommuting()):
    ansatz = hea()
    pa = PauliAveraging(operator=Hfull, ket=ansatz, n_shots=qarp.EXACT, grouping=strategy)
    engine = QarpEngine()
    engine.build([pa])
    value = engine.run(params_for(ansatz))[0]
    print(f"{strategy!r:24} {len(pa.sub_blocks):2d} circuits   <H> = {value:+.12f}")

## 3. Grouping in time evolution — `TrotterBlock` and symmetry

Grouping is not only about circuit counts. `TrotterBlock(grouping=...)` exponentiates each group's (commuting) sum *exactly*, so which terms share a group changes the physics of the Trotter error.

The JW-mapped hopping Hamiltonian is the canonical case: each bond term `(X_i X_{i+1} + Y_i Y_{i+1})/2` conserves particle number, but its `XX` and `YY` halves individually do not. Split them into different exponentials and the Trotter error *leaks probability between particle-number sectors*. We start in |0011⟩ (2 particles) and measure the exact probability of ending outside the 2-particle sector.

In [ ]:
n = 4
Hbond = QubitOperator()
for i in range(n - 1):  # bond-wise term order: XX, YY adjacent per bond
    Hbond += QubitOperator(f"X{i} X{i+1}", 0.5)
    Hbond += QubitOperator(f"Y{i} Y{i+1}", 0.5)

def sector_leakage(H, strategy, n_particles=2):
    circ = CompositeBlock([ComputationalBasisStateBlock([0, 0, 1, 1]),
                           TrotterBlock(n, H, steps=2, time=1.0, grouping=strategy)])
    circ.measure([(q, q) for q in range(n)])
    circ.build()
    sampler = Sampler(ket=circ, n_shots=qarp.EXACT)
    engine = QarpEngine()
    engine.build([sampler])
    dist = engine.run()[0]
    return sum(p for bits, p in dist.items() if sum(bits) != n_particles)

bond_terms = [dict(t) for t in Hbond.terms if t]
bond_labels = [" ".join(f"{p}{q}" for q, p in sorted(t.items())) for t in bond_terms]
for strategy in (NoGrouping(), QubitWiseCommuting(), FullyCommuting()):
    groups = [[bond_labels[i] for i in g] for g in strategy.group(bond_terms, n)]
    print(f"{strategy!r:24} leakage = {sector_leakage(Hbond, strategy):.3e}   groups: {groups}")

`QubitWiseCommuting` regroups the terms into an all-`XX` set and an all-`YY` set — QWC is a *measurement* criterion, and here it actively splits the symmetry-preserving pairs even though the term order kept them adjacent. `FullyCommuting` groups whole bonds and conserves particle number exactly; `NoGrouping` happens to conserve it too, but only because the insertion order left each bond's halves adjacent (adjacent commuting factors merge). That luck runs out if the operator is assembled in a different order:

In [ ]:
Hshuffled = QubitOperator()
for i in range(n - 1):  # all XX first, then all YY
    Hshuffled += QubitOperator(f"X{i} X{i+1}", 0.5)
for i in range(n - 1):
    Hshuffled += QubitOperator(f"Y{i} Y{i+1}", 0.5)

for strategy in (NoGrouping(), QubitWiseCommuting(), FullyCommuting()):
    print(f"{strategy!r:24} leakage = {sector_leakage(Hshuffled, strategy):.3e}")

With the shuffled order even greedy `FullyCommuting` lands on the all-`XX` / all-`YY` split (greedy first-fit is deterministic but term-order sensitive). When a conservation law matters, encode it in the grouping itself — which is what custom strategies are for.

## 4. Writing a custom strategy

Subclass `GroupingStrategy` and implement `group(terms, n_qubits) -> list[list[int]]`. The contract:

- return an index *partition* (every index in exactly one group), deterministic given the input order;
- strategies never see coefficients — those stay positionally attached in the consumer;
- set `qubit_wise = True` only if every group is guaranteed qubit-wise commuting;
- you are responsible for the guarantee your consumer needs (within-group commutation here — same-support Pauli pairs like `XX`/`YY` always commute, but e.g. `X0`/`Y0` would not, so this toy strategy is only valid for operators like our hopping chain).

This strategy groups terms by their qubit support, pinning each bond's `XX`/`YY` pair together regardless of term order:

In [ ]:
class SameSupport(GroupingStrategy):
    """Group terms acting on the same set of qubits (valid when same-support
    terms commute, e.g. JW hopping XX/YY pairs)."""

    qubit_wise = False

    def group(self, terms, n_qubits):
        by_support = {}
        for i, t in enumerate(terms):
            by_support.setdefault(frozenset(t), []).append(i)
        return list(by_support.values())


for H, label in ((Hbond, "bond-ordered"), (Hshuffled, "shuffled")):
    print(f"SameSupport() on {label:13} H: leakage = {sector_leakage(H, SameSupport()):.3e}")

**Defaults across qarp:** `PauliAveraging` and the Trotter family default to `FullyCommuting`; circuit cutting requires a `qubit_wise` strategy and uses `QubitWiseCommuting`. The module also exposes the building blocks (`greedy_first_fit`, `qubit_wise_commute`, `group_basis`, `diagonalise_group`) for strategies that need them — see `qarp/operators/grouping.py`.